# Reproducing Peterson, Singh et al. (2021) from dandiset 000055 (AJILE12)

**Paper.** Peterson SM, Singh SH, Wang NXR, Rao RPN, Brunton BW (2021). Behavioral and neural
variability of naturalistic arm movements. *eNeuro* 8(3).
[doi:10.1523/ENEURO.0007-21.2021](https://doi.org/10.1523/ENEURO.0007-21.2021). This attempt
used the full text (open access, PubMed Central PMC8225404). The Extended Data figures are EPS
files and were not read; claims on them rest on their legends and the Results.

**Dandiset.** [DANDI:000055](https://dandiarchive.org/dandiset/000055/0.220127.0436), "AJILE12",
version 0.220127.0436. The paper cites no DANDI version: it points to a curated dataset on
Figshare, and the dandiset was published later and names the paper as describing it. This is
the only published version. The draft has the same 55 assets and size; its modification date
is later, and the analysis does not depend on anything it could have changed.

**What the dandiset holds.** One NWB file per subject and recording day (12 subjects, days 3 to
7 after implantation, 55 files): ECoG at 500 Hz, pose of nine upper-body keypoints at 30 Hz,
coarse behavioural labels, and the wrist movement events with nine stored features. It holds
the whole recordings, not the paper's curated 10 s segments.

**Outside input.** The paper's projection matrices from electrodes to AAL regions and the
regions' centroids, from Figshare
([doi:10.6084/m9.figshare.12114408.v1](https://doi.org/10.6084/m9.figshare.12114408.v1)),
read into memory. Claims marked "with inputs" use them and depend on `projection-link`.

**Code read.** The paper's analysis code, GitHub `BruntonUWBio/naturalistic_arm_movements_ecog`
at commit 04c6faf2f219586fa834d10c614ef48a694cec55: spectral power (`compute_spectral_power.py`,
`tfr_utils.py`, `config.py`), projection (`project_ecog_to_rois.m`), Figure 2 features
(`feat_plot_utils.py`), Figure 5 (`Plot_banded_power_Fig5.ipynb`) and the regression
(`regression_utils.py`). Choices it settles are listed in the helper's docstring.

**Random numbers.** Each section that draws them has its own seeded generator.

| Claim | Data | Status |
|---|---|---|
| subjects-days-hemispheres | dandiset | tested |
| demographics | dandiset | blocked: no sex or age in the dandiset |
| electrode-counts | dandiset | tested |
| bad-electrodes | dandiset | tested |
| event-counts | dandiset | tested |
| features-differ-subjects | dandiset | tested |
| days-one-dominant | dandiset | tested |
| duration-magnitude-small | dandiset | tested |
| angle-bimodal | dandiset | tested |
| feature-correlations | dandiset | tested |
| bimanual-skew | dandiset | tested |
| projection-link | with inputs | tested |
| group-pattern-rois | with inputs | tested |
| precentral-subject-ci | with inputs | tested |
| precentral-subjects-differ | with inputs | tested |
| precentral-day-effects | with inputs | tested |
| baseline-reduces-day-variability | with inputs | tested |
| regression-r2-max | with inputs | tested |
| regression-r2-sensorimotor | with inputs | tested |
| regression-angle-day-top | with inputs | tested |
| regression-angle-coefficient | with inputs | tested |
| regression-angle-retained | with inputs | tested |

In [ ]:
import os
import time
import warnings

os.environ.setdefault("OMP_NUM_THREADS", "1")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import ajile

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

nb_started = started = time.time()
sessions = ajile.load_all()
dandiset_meta = ajile.dandiset_metadata()
proj, centroids = ajile.projection_matrices()
ev = ajile.reach_features(sessions)
subjects = sorted({s["subject"] for s in sessions})
first = {}
for s in sessions:
    first.setdefault(s["subject"], s)
print(f"loaded {len(sessions)} files of dandiset {ajile.DANDISET_ID} version {ajile.DANDISET_VERSION}, "
      f"{len(ev)} events, projection matrices for {len(proj)} subjects, in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
files = pd.DataFrame([{
    "subject": s["subject"], "day": s["day"], "session_start": s["session_start"],
    "ecog_hours": round(s["ecog_shape"][0] / s["ecog_rate"] / 3600, 2), "channels": s["ecog_shape"][1],
    "ecog_rate": s["ecog_rate"], "chunk": s["ecog_chunks"], "pose_hours": round(s["pose_n"] / s["pose_rate"] / 3600, 2),
    "events": len(s["reaches"]), "event_wrist": s["event_wrist"], "hemisphere": s["hemisphere"],
    "offset_h": round(s["ecog_offset"] / 3600, 3)} for s in sessions])
print(files.to_string(index=False))
print("\nArchive summary:", {k: dandiset_meta["assetsSummary"][k] for k in ("numberOfSubjects", "numberOfFiles", "numberOfBytes")},
      "species:", [x["name"] for x in dandiset_meta["assetsSummary"]["species"]])
print("Subject fields in the files:", sorted({tuple(s["subject_fields"]) for s in sessions}))
print("Species in the files:", sorted({s["species"] for s in sessions}))
print("Dandiset metadata mentions sex or age:", any(k in str(dandiset_meta).lower() for k in ("sex", "\"age\"", "agereference")))
print("ECoG unit, conversion:", sorted({(s["ecog_unit"], s["ecog_conversion"]) for s in sessions}),
      "; pose unit, conversion:", sorted({(s["pose_unit"], str(s["pose_conversion"])) for s in sessions}))
print("ECoG dtype and filter pipeline:", sorted({(s["ecog_dtype"], s["ecog_filters"]) for s in sessions}))
print("ECoG starting_time:", sorted({s["ecog_start"] for s in sessions}), "; pose rate:", sorted({s["pose_rate"] for s in sessions}))
print("Other acquisition series, by subject:", {s["subject"]: sorted(k for k in s["acquisition"] if k != "ElectricalSeries")
                                                for s in sessions if len(s["acquisition"]) > 1})
print("Their descriptions:", sorted({(k, v) for s in sessions for k, v in s["acquisition"].items() if k != "ElectricalSeries"}))
mismatch = []
for s in sessions:
    d = s["event_times"] - s["reaches"].start_time.to_numpy()
    off = np.abs(d) > 0.01
    if off.any():
        mismatch.append({"file": f"{s['subject']} day {s['day']}", "events": len(d), "differing": int(off.sum()),
                         "difference_s": np.unique(np.round(d[off], 1)).tolist()[:3], "ecog_offset_s": round(s["ecog_offset"], 1),
                         "start_time_of_differing": np.round(s["reaches"].start_time.to_numpy()[off][:3], 1).tolist()})
print("Files where ReachEvents timestamps differ from the reaches table's start_time by more than 0.01 s:")
print(pd.DataFrame(mismatch).to_string(index=False))

The archive summary lists 12 subjects, all human. The files give only species and subject id:
no sex or age, so the paper's demographics cannot be checked. Every file holds 24 h of pose and
15.5 to 24 h of ECoG. ECoG data are stated in volts with a conversion of 1e-6, so the stored
values are microvolts. The pose is stated in meters with a conversion of NaN, but its values are
image pixels (up to about 1233 by 708 in the windows read below; the reach features are in
pixels). The ECG series (`ECGL`, `ECGR`, subject 01 only) carry the description of the EOG
series ("Electrooculography for tracking saccades"). ECoG samples are little-endian float32 with gzip
(`deflate`) as the only filter, in every file. The ECoG chunk length is 84375 samples except in
the three files that start later in the day (54412, 55031 and 55820 samples).

The `ReachEvents` timestamps equal the reaches table's `start_time` except in four files. In the
three files whose ECoG starts later in the day, every timestamp is earlier by the file's offset
of `session_start_time` from the start of its day (8.1 to 8.5 h). In subject 08 day 4, one
event's `start_time` is 86440.3 s, past the end of the 24 h file, where its timestamp is 40.3 s.

In [ ]:
el_rows = []
for s in sessions:
    e = s["electrodes"]
    depth = e.group_name.map(lambda g: "depth" in s["groups"].get(g, "").lower())
    el_rows.append({"subject": s["subject"], "day": s["day"], "electrodes": len(e),
                    "ecog_channels": s["ecog_shape"][1], "depth": int(depth.sum()),
                    "no_xyz": int(e.x.isna().sum()), "bad": int((~e.good).sum()),
                    "bad_depth": int((~e.good & depth).sum()),
                    "locations": ",".join(sorted(set(e.location))), "imp_nan": bool(e.imp.isna().all()),
                    "R2_lo_max": round(float(e.low_freq_R2.max()), 3), "R2_hi_max": round(float(e.high_freq_R2.max()), 3),
                    "R2_nan": int(e.low_freq_R2.isna().sum())})
el_tab = pd.DataFrame(el_rows)
print(el_tab.to_string(index=False))
print("\nGroups and their descriptions, per subject (first file):")
for sub in subjects:
    s = first[sub]
    counts = s["electrodes"].group_name.value_counts().to_dict()
    print(sub, {g: (d, counts.get(g, 0)) for g, d in s["groups"].items()})
same_r2 = all(np.allclose(s["electrodes"].low_freq_R2.fillna(9), first[s["subject"]]["electrodes"].low_freq_R2.fillna(9))
              for s in sessions)
print("\nStored R2 columns identical across a subject's files:", same_r2)
print("Electrode coordinates identical across a subject's files:",
      all(np.allclose(s["electrodes"][["x", "y", "z"]].fillna(0), first[s["subject"]]["electrodes"][["x", "y", "z"]].fillna(0)) for s in sessions))

Electrode `location` is "unknown" and impedance is missing for every electrode. Subject 03 has
16 depth electrodes (groups LAD, LPD) without coordinates, and these have no stored R2. In
subject 12 the 8 electrodes without stored R2 (rows 116 to 123, group PHD) do have coordinates.
The stored R2 columns, the coordinates and the electrode list are the same in every file of a
subject; the `good` flags differ between days.

In [ ]:
reach_desc = first["01"]["reach_descriptions"]
for k, v in reach_desc.items():
    print(f"{k}: {v!r}")
print()
summary = ev[["duration", "magnitude", "angle_stored", "onset_speed", "speech_ratio", "bimanual_ratio",
              "bimanual_overlap", "bimanual_class", "time_of_day"]].describe(percentiles=[0.01, 0.5, 0.99]).T
print(summary.round(3).to_string())
print("\nBimanual ratio missing (set to 0.5 as the code does):", int(ev.bimanual_ratio_nan.sum()))
print("Events with duration below 0.5 s:", int((ev.duration < 0.5).sum()), "; above 4 s:", int((ev.duration > 4).sum()),
      "(Methods keep 0.5 to 4 s)")
print("Bimanual overlap above the reach duration:", int((ev.bimanual_overlap > ev.duration + 1e-6).sum()),
      "; overlap above 1:", int((ev.bimanual_overlap > 1).sum()),
      "; median overlap / duration where overlap > 0:", round(float((ev.bimanual_overlap / ev.duration)[ev.bimanual_overlap > 0].median()), 3))
print("Bimanual class values:", sorted(ev.bimanual_class.unique()))
print("Largest number of events in one file:", int(files.events.max()), "(Methods: up to 200 per day)")

Speech ratio lies within 0 to 1, as described. `Bimanual_overlap` is described as a fraction of
the movement duration, but reaches 3.83 (1708 events above 1) and is never above the reach
duration: it is in seconds, as in the paper's code (overlap in frames divided by 30). The stored
reach angle covers -180 to 180 degrees; its convention is checked against the pose below. 328
durations (`stop_time - start_time`) are below the 0.5 s the Methods keep (the shortest is
0.23 s). No file has more than 200 events, the Methods' cap per day.

In [ ]:
# Pose clock and reach angle convention: contralateral wrist around 20 events per file
# (seeded), at the stored event times and 30 s later.
rng_pose = np.random.default_rng(55002)
pose_tasks, pose_events = [], []
for s in sessions:
    r = s["reaches"]
    idx = np.sort(rng_pose.choice(len(r), min(20, len(r)), replace=False))
    kp = "R_Wrist" if s["event_wrist"] == "r_wrist" else "L_Wrist"
    t = r.start_time.to_numpy()[idx]
    pose_tasks.append({"url": s["url"], "keypoint": kp, "times": np.r_[t, t + 30.0]})
    pose_events.append((s, idx))
started = time.time()
wrist = ajile.run_wrist(pose_tasks)
n0 = int(1.5 * ajile.FS_VIDEO)
rows = []
for (s, idx), w in zip(pose_events, wrist):
    n = len(idx)
    speed = np.hypot(*np.diff(w, axis=1).transpose(2, 0, 1)) * ajile.FS_VIDEO   # px/s
    resp = np.nanmean(speed[:, n0:n0 + 15], axis=1) - np.nanmean(speed[:, 0:15], axis=1)
    r = s["reaches"].iloc[idx]
    for k in range(n):
        seg = w[k, n0:n0 + int(round(r.stop_time.iloc[k] * 30)) - int(round(r.start_time.iloc[k] * 30)) + 1] - w[k, n0]
        rad = np.hypot(seg[:, 0], seg[:, 1])
        j = np.nanargmax(rad) if np.isfinite(rad).any() else 0
        ang = np.degrees(np.arctan2(seg[j, 1], seg[j, 0]))
        rows.append({"subject": s["subject"], "day": s["day"], "resp_true": resp[k], "resp_shift": resp[n + k],
                     "angle_pose": ang, "angle_stored": r.Reach_angle_degrees.iloc[k],
                     "mag_pose": rad[j], "mag_stored": r.Reach_magnitude_px.iloc[k]})
pose_chk = pd.DataFrame(rows)
d_ang = (pose_chk.angle_pose - pose_chk.angle_stored + 180) % 360 - 180
print(f"wrist windows read in {time.time() - started:.0f} s")
print("Wrist speed 0 to 0.5 s minus -1.5 to -1 s (px/s), median over events, per subject:")
print(pose_chk.groupby("subject")[["resp_true", "resp_shift"]].median().round(1).T.to_string())
print("Events with a higher speed after the event time than before: true",
      round(float((pose_chk.resp_true > 0).mean()), 3), "; shifted by 30 s", round(float((pose_chk.resp_shift > 0).mean()), 3))
print("Median |stored angle - pose angle| (image coordinates, y down):", round(float(np.nanmedian(np.abs(d_ang))), 1),
      "deg; within 20 deg:", round(float(np.nanmean(np.abs(d_ang) < 20)), 3))
print("Same with the pose angle taken with y up:",
      round(float(np.nanmedian(np.abs((-pose_chk.angle_pose - pose_chk.angle_stored + 180) % 360 - 180))), 1), "deg")
print("Spearman r, pose and stored magnitude:", round(stats.spearmanr(pose_chk.mag_pose, pose_chk.mag_stored, nan_policy="omit")[0], 3))
late_files = [f"{s['subject']} day {s['day']}" for s in sessions if s["ecog_offset"] > 0]
pose_chk["file"] = pose_chk.subject + " day " + pose_chk.day.astype(str)
print("Late-starting files:", pose_chk[pose_chk.file.isin(late_files)].groupby("file")[["resp_true", "resp_shift"]].median().round(1).to_dict("index"))
print("Pose x, y ranges (px), over these windows:", np.round(np.nanpercentile(np.concatenate(wrist).reshape(-1, 2), [0, 100], axis=0), 1).tolist())

The wrist speeds up after the stored event times (median 27.8 to 206.9 px/s per subject) and
not 30 s later (-2.0 to 1.8 px/s), and in 94% of events against 49%, so the pose and the event
table share a clock, also in the three files that start later in the day. The stored reach angle is the angle of the wrist's largest displacement in image
coordinates, with y pointing down: a positive stored angle is a downward reach. The paper's
code negates it before folding, so that upward reaches are positive; the description of the
column does not give the convention.

In [ ]:
sleep = np.zeros(24)
ev_hours = np.histogram(ev.t / 3600, bins=np.arange(25))[0]
for s in sessions:
    e = s["epochs"][s["epochs"].label.str.contains("Sleep")]
    for a, b in zip(e.start, e.stop):
        for h in range(24):
            sleep[h] += max(0.0, min(b, (h + 1) * 3600) - max(a, h * 3600)) / 3600
print("Hours labelled Sleep/rest, by hour after the start of the file's day:")
print(np.round(sleep, 1).tolist())
print("Events by hour after the start of the file's day:")
print(ev_hours.tolist())
print("Epoch labels:", sorted({l for s in sessions for l in s["epochs"].label.unique()}))
print("Last epoch stop (h), per file:", sorted({round(s["epochs"].stop.max() / 3600, 2) for s in sessions}))

Sleep labels peak in the first six hours after the start of the file's day and events are
fewest from 2 to 5 h. This is consistent with each file's day starting at local midnight. The
`session_start_time` of the full-day files is 19:00 at UTC-05:00, which would put sleep at
19:00 to 01:00; the stated time zone does not fit the labels (consistent with the data, not
shown). Time of day is therefore taken as hours from the start of the file's day.

The epochs of every file run to 24 h, also in the three files whose ECoG covers only 15.5 to
15.9 h and whose `session_start_time` is 8.1 to 8.5 h after the day's start (subject 01 day 7,
04 day 3, 06 day 3). The ECoG `starting_time` is 0 in every file, so the files place the ECoG
at the start of the day, as the pose and the reaches table. The `ReachEvents` timestamps of
these files, and the ECoG check below, show that the ECoG starts at `session_start_time`.

In [ ]:
# Recordings deposited twice: compare per-electrode SD and kurtosis (computed over the whole
# recording and stored per file), the event tables and the event counts between files.
dups = []
for i, a in enumerate(sessions):
    for b in sessions[i + 1:]:
        same_sd = (len(a["electrodes"]) == len(b["electrodes"]) and
                   np.allclose(a["electrodes"].standard_deviation, b["electrodes"].standard_deviation))
        same_ev = (len(a["reaches"]) == len(b["reaches"]) and
                   np.allclose(a["reaches"].to_numpy(float), b["reaches"].to_numpy(float), equal_nan=True))
        if same_sd or same_ev:
            dups.append((a["path"], b["path"], same_sd, same_ev))
print("File pairs with identical electrode SDs or identical event tables:", dups if dups else "none")

No two files share their per-electrode statistics or their event tables, so no recording is
deposited twice.

In [ ]:
# Reader check: the direct chunk-index reader (ajile.fetch_windows) against h5py, for two
# channels and two windows in three files: the file with the shorter chunk length and two
# drawn at random. One window straddles a chunk boundary.
import h5py
import remfile
rng_reader = np.random.default_rng(55006)
odd = [i for i, s in enumerate(sessions) if s["ecog_chunks"][0] != 84375]
others = [i for i in range(len(sessions)) if i not in odd]
rows = []
for i in odd[:1] + sorted(rng_reader.choice(others, 2, replace=False).tolist()):
    s = sessions[i]
    per = s["ecog_chunks"][0]
    k = int(rng_reader.integers(1, s["ecog_shape"][0] // per))
    times = np.array([(k * per - 300) / ajile.FS - ajile.WINDOW[0],
                      float(rng_reader.uniform(10, s["ecog_shape"][0] / ajile.FS - 10))])
    channels = np.sort(rng_reader.choice(s["ecog_shape"][1], 2, replace=False))
    mine = ajile.fetch_windows(s["url"], s["ecog_shape"][0], times, channels)
    _, index = ajile.chunk_index(s["url"])
    starts = np.round(times * ajile.FS).astype(int) - int(round(-ajile.WINDOW[0] * ajile.FS))
    with h5py.File(remfile.File(s["url"]), "r") as f:
        d = f["acquisition/ElectricalSeries/data"]
        for a, st in enumerate(starts):
            for b, ch in enumerate(channels):
                ref = d[st:st + mine.shape[2], int(ch)]
                rows.append({"file": f"{s['subject']} day {s['day']}", "start_sample": int(st),
                             "crosses_chunk": st // per != (st + mine.shape[2] - 1) // per, "channel": int(ch),
                             "identical": bool(np.array_equal(ref, mine[a, b])),
                             "max_abs_diff_uV": float(np.max(np.abs(ref - mine[a, b]))),
                             "chunks_in_index": len(index),
                             "chunks_without_gzip": sum(m & 1 for _, _, m in index.values())})
print(pd.DataFrame(rows).to_string(index=False))

The direct reader returns the same samples as h5py in all 12 comparisons (three files, two
channels and two windows each; largest difference 0 microvolts), including 8 windows that
straddle a chunk boundary and subject 01 day 7, whose chunks are shorter. No chunk in these
three files is stored without gzip.

## ECoG around the events

For every event, ECoG from 2 s before to 2 s after the event is streamed for the electrodes
that carry 95% of the subject's precentral weight (set P), and for a seeded subsample of five
events per file, at the event and 30 s later, for the electrodes that carry 99% of every
region's weight (set A). In the three files above, the event times are moved onto the ECoG
clock by subtracting the offset of `session_start_time` from the day's start; the check after
the loading cell shows this is needed.

In [ ]:
rng_sub = np.random.default_rng(55001)
w_all, tasks_p, tasks_a, sub_idx = {}, [], [], []
for s in sessions:
    sub = s["subject"]
    n_el = len(s["electrodes"])
    w = ajile.roi_weights(proj[sub], n_el, first[sub]["electrodes"].good.to_numpy())
    w_all[sub] = w
    t = s["reaches"].start_time.to_numpy() - s["ecog_offset"]
    ch_p = ajile.covering(w[:, :1], 0.95)
    tasks_p.append({"url": s["url"], "n_samples": s["ecog_shape"][0], "times": t, "channels": ch_p,
                    "weights": w[ch_p, :1] / w[ch_p, :1].sum(axis=0)})
    ch_a = ajile.covering(w, 0.99)
    k = np.sort(rng_sub.choice(len(t), min(5, len(t)), replace=False))
    sub_idx.append(k)
    trunc = np.zeros((len(ch_a), 1))
    trunc[np.isin(ch_a, ch_p), 0] = w[ch_p, 0] / w[ch_p, 0].sum()
    tasks_a.append({"url": s["url"], "n_samples": s["ecog_shape"][0], "times": np.r_[t[k], t[k] + 30.0],
                    "channels": ch_a, "weights": np.c_[w[ch_a] / w[ch_a].sum(axis=0), trunc]})
started = time.time()
out_p = ajile.run_power(tasks_p)
print(f"set P: {sum(len(t['times']) for t in tasks_p)} events x {sum(len(t['channels']) for t in tasks_p) / len(tasks_p):.1f} "
      f"electrodes on average, in {time.time() - started:.0f} s")
started = time.time()
out_a = ajile.run_power(tasks_a)
print(f"set A: {sum(len(k) for k in sub_idx)} events (and as many shifted windows) x "
      f"{sum(len(t['channels']) for t in tasks_a) / len(tasks_a):.1f} electrodes on average, in {time.time() - started:.0f} s")

In [ ]:
# One row per event with its precentral band power (set P) and the rejection flags.
pieces = []
for s, task, o in zip(sessions, tasks_p, out_p):
    b = ajile.band_values(o["roi_tfr"][:, 0])
    braw = ajile.band_values(o["roi_tfr_raw"][:, 0])
    pieces.append(pd.DataFrame({"subject": s["subject"], "day": s["day"], "event": np.arange(len(task["times"])),
                                "lfb": b[:, 0], "hfb": b[:, 1], "lfb_raw": braw[:, 0], "hfb_raw": braw[:, 1],
                                "bad_window": o["bad_window"], "seg_power": o["seg_power"], "hf_noise": o["hf_noise"]}))
pw = pd.concat(pieces, ignore_index=True)
assert len(pw) == len(ev) and (pw[["subject", "day", "event"]].to_numpy() == ev[["subject", "day", "event"]].to_numpy()).all()
pw["low_power"] = ~pw.bad_window & (pw.seg_power < 0)
ok = ~pw.bad_window & ~pw.low_power
mu = pw[ok].groupby("subject").hf_noise.mean()
sd = pw[ok].groupby("subject").hf_noise.std()
pw["hf_outlier"] = ok & (pw.hf_noise > pw.subject.map(mu) + 3 * pw.subject.map(sd))
pw["keep"] = ok & ~pw.hf_outlier

In [ ]:
print("Events dropped per subject: missing or flat data, mean power below 0 dB, 115-125 Hz outlier; kept")
drop = pw.groupby("subject").agg(events=("keep", "size"), missing_or_flat=("bad_window", "sum"),
                                 below_0dB=("low_power", "sum"), hf_outlier=("hf_outlier", "sum"), kept=("keep", "sum"))
print(drop.to_string())
print("Total kept:", int(pw.keep.sum()), "of", len(pw))
print("Electrodes in set P per subject:", {s["subject"]: len(t["channels"]) for s, t in zip(sessions, tasks_p)})
print("Mean dB power (microvolts^2) of kept windows, 1st to 99th percentile:", np.round(np.percentile(pw.seg_power[pw.keep], [1, 99]), 1).tolist())
print("Precentral LFB and HFB of kept events (dB), 1st to 99th percentile:",
      np.round(np.percentile(pw.lfb[pw.keep], [1, 99]), 2).tolist(), np.round(np.percentile(pw.hfb[pw.keep], [1, 99]), 2).tolist())

124 of 8088 events (1.5%) are dropped: 89 whose window holds missing data, lies outside the
recording or has a channel of zeros, and 35 with outlying power at 115 to 125 Hz. No window has
a mean power below 0 dB.

In [ ]:
# Clock check. (1) The three late-starting files: precentral response at the corrected times,
# at the stored times, and 30 s after each. (2) Every subject: response at the event times
# against 30 s later (set A subsample, precentral projection), as a contrast between true and
# shifted labels.
late = [i for i, s in enumerate(sessions) if s["ecog_offset"] > 0]
clock_tasks, clock_rows = [], []
for i in late:
    s = sessions[i]
    t = s["reaches"].start_time.to_numpy()
    for name, tt in (("stored times", t), ("corrected times", t - s["ecog_offset"]),
                     ("stored + 30 s", t + 30), ("corrected + 30 s", t - s["ecog_offset"] + 30)):
        inside = np.flatnonzero((tt > 3) & (tt < s["ecog_shape"][0] / ajile.FS - 3))[:40]
        clock_tasks.append({**tasks_p[i], "times": tt[inside]})
        clock_rows.append({"file": f"{s['subject']} day {s['day']}", "times": name, "events": len(inside)})
rows = []
for r, o in zip(clock_rows, ajile.run_power(clock_tasks)):
    b = ajile.band_values(o["roi_tfr"][:, 0])
    rows.append({**r, "LFB": np.median(b[:, 0]), "HFB": np.median(b[:, 1])})
print(pd.DataFrame(rows).round(2).to_string(index=False))

rows = []
for s, k, o in zip(sessions, sub_idx, out_a):
    b = ajile.band_values(o["roi_tfr"][:, 0])
    n = len(k)
    good = ~o["bad_window"][:n] & ~o["bad_window"][n:]
    rows.append(pd.DataFrame({"subject": s["subject"], "lfb_true": b[:n, 0], "lfb_shift": b[n:, 0],
                              "hfb_true": b[:n, 1], "hfb_shift": b[n:, 1]})[good])
shift = pd.concat(rows)
shift["contrast_true"] = shift.hfb_true - shift.lfb_true
shift["contrast_shift"] = shift.hfb_shift - shift.lfb_shift
tab = shift.groupby("subject")[["contrast_true", "contrast_shift"]].median()
tab["events"] = shift.groupby("subject").size()
tab["wilcoxon_p"] = [stats.wilcoxon(g.contrast_true, g.contrast_shift)[1] for _, g in shift.groupby("subject")]
print("\nPrecentral HFB minus LFB (dB), median over subsampled events, at the event and 30 s later:")
print(tab.round(3).to_string())

In subject 01 day 7 and subject 06 day 3, the stored times give an LFB increase (0.61 and
0.70 dB), and the corrected times the LFB decrease with HFB increase seen in the other files
(-1.40 and 0.12 dB; -1.47 and 2.77 dB). With the `ReachEvents` timestamps, which equal the
corrected times, this shows that in these files the ECoG starts at `session_start_time` while
the reaches table counts from the day's start; the corrected times are used throughout. In
subject 04 day 3 neither timing gives a clear response (corrected -0.29 and -0.10 dB, from 35
events; stored 0.40 and -0.14 dB, from 13), so its alignment could not be established from the
ECoG; the same correction is applied, as its `ReachEvents` timestamps also support.

Across subjects, precentral HFB minus LFB is larger at the event times than 30 s later in 10
of 12 subjects, significantly in 5 (05, 06, 07, 08, 11; Wilcoxon p below 0.05) on 15 to 25
events each. In subjects 01 and 02 it is not larger, so this subsample does not establish the
ECoG clock for them; it is consistent with a shared clock elsewhere. The shifted windows alone
give medians from -1.42 to 1.39 dB, so with 15 to 25 events this check has little power in any
one subject.

## subjects-days-hemispheres

> The data come from 12 subjects, each recorded for 3 to 5 days within days 3 to 7 after
> implantation, with electrodes in the right hemisphere in five subjects and in the left in
> seven; events are movements of the wrist contralateral to the implant.

In [ ]:
per_sub = files.groupby("subject").agg(days=("day", lambda d: sorted(d)), n_days=("day", "size"),
                                       hemisphere=("hemisphere", "first"), wrist=("event_wrist", lambda w: sorted(set(w))))
per_sub["contralateral"] = [(h == "L" and w == ["r_wrist"]) or (h == "R" and w == ["l_wrist"])
                            for h, w in zip(per_sub.hemisphere, per_sub.wrist)]
print(per_sub.to_string())
print("\nSubjects:", len(per_sub), "; days per subject:", sorted(per_sub.n_days.unique()),
      "; all days within 3 to 7:", bool(files.day.between(3, 7).all()))
print("Right hemisphere:", int((per_sub.hemisphere == "R").sum()), "; left:", int((per_sub.hemisphere == "L").sum()),
      "; event wrist contralateral in every subject:", bool(per_sub.contralateral.all()))

**Result: reproduced.** The dandiset holds 12 subjects with 3 to 5 files each, all on days 3 to
7. Five subjects have electrodes in the right hemisphere and seven in the left, and in every
subject the events are of the contralateral wrist.

## demographics

> The subjects were 8 males and 4 females, aged 29.4 +/- 7.9 years.

Blocked: the files hold only species and subject id, and the dandiset metadata has no sex or
age (data check).

## electrode-counts

> Subjects had 87.0 +/- 12.9 surface electrodes, and five subjects also had 23.2 +/- 12.1 depth
> electrodes.

In [ ]:
def electrode_counts(drop_no_xyz):
    rows = []
    for sub in subjects:
        s = first[sub]
        e = s["electrodes"]
        depth = e.group_name.map(lambda g: "depth" in s["groups"].get(g, "").lower()).to_numpy()
        keep = e.x.notna().to_numpy() if drop_no_xyz else np.ones(len(e), bool)
        rows.append({"subject": sub, "surface": int((~depth & keep).sum()), "depth": int((depth & keep).sum())})
    return pd.DataFrame(rows).set_index("subject")

for label, drop_no_xyz in (("(a) all electrodes", False), ("(b) electrodes with coordinates", True)):
    c = electrode_counts(drop_no_xyz)
    d = c.depth[c.depth > 0]
    print(f"{label}: surface {c.surface.mean():.1f} +/- {c.surface.std():.1f}; subjects with depth {len(d)} "
          f"({', '.join(d.index)}); depth {d.mean():.1f} +/- {d.std():.1f}")
    print("   per subject:", c.T.to_dict())
print("Paper: surface 87.0 +/- 12.9; 5 subjects with depth 23.2 +/- 12.1")

**Result: partially reproduced.** Under both readings the mean number of surface electrodes,
84.8, is within tolerance of 87.0, and the mean number of depth electrodes, 23.7, within
tolerance of 23.2. But 7 subjects (reading a) or 6 (reading b) have depth electrodes, not 5.

## bad-electrodes

> Removing electrodes with abnormal SD or kurtosis left out 4.9 +/- 4.9 surface electrodes per
> subject, and 1.0 +/- 1.4 depth electrodes in each of the five subjects with depth electrodes.

In [ ]:
rows = []
for sub in subjects:
    ss = [s for s in sessions if s["subject"] == sub]
    s0 = ss[0]
    depth = s0["electrodes"].group_name.map(lambda g: "depth" in s0["groups"].get(g, "").lower()).to_numpy()
    bad = np.array([~s["electrodes"].good.to_numpy() for s in ss])
    for reading, b in (("(a) first file", bad[0].astype(float)), ("(b) any file", bad.any(axis=0).astype(float)),
                       ("(c) mean over files", bad.mean(axis=0))):
        rows.append({"subject": sub, "reading": reading, "surface": float(b[~depth].sum()),
                     "depth": float(b[depth].sum()) if depth.any() else np.nan})
bad_tab = pd.DataFrame(rows)
print(bad_tab.pivot(index="subject", columns="reading", values=["surface", "depth"]).round(2).to_string())
for reading, g in bad_tab.groupby("reading"):
    print(f"{reading}: surface {g.surface.mean():.2f} +/- {g.surface.std():.2f}; "
          f"depth {g.depth.mean():.2f} +/- {g.depth.std():.2f} over {g.depth.notna().sum()} subjects")
print("Paper: surface 4.9 +/- 4.9; depth 1.0 +/- 1.4 over 5 subjects")
a = bad_tab[(bad_tab.reading == "(a) first file") & (bad_tab.subject != "03")]
print(f"Reading (a) without subject 03 (whose 16 depth electrodes have no coordinates): depth {a.depth.mean():.2f} over {a.depth.notna().sum()} subjects")

**Result: partially reproduced.** Under reading (a), the paper's code's choice, 4.83 +/- 4.99
surface electrodes per subject are flagged bad, matching 4.9 +/- 4.9, but 4.29 depth electrodes
per subject with depth electrodes, against 1.0. Without subject 03, whose 16 depth electrodes
have no coordinates and are all flagged, the depth mean is 2.33. Readings (b) and (c) match
neither value.

## event-counts

> The total number of movement events per subject was between 151 and 947, with a median of
> 640 across subjects.

In [ ]:
counts = pd.DataFrame({"all_events": ev.groupby("subject").size(), "after_rejection": pw[pw.keep].groupby("subject").size()})
print(counts.T.to_string())
for c in counts:
    print(f"{c}: min {counts[c].min()}, max {counts[c].max()}, median {counts[c].median():.1f}")
print("Paper: min 151, max 947, median 640")
print("Events per file at the Methods' cap of 200 or one below:", int((files.events >= 199).sum()), "of", len(files))
c = counts.all_events * (1 - 0.178)
print(f"Variant, all events less the Methods' mean 17.8% of false positives: min {c.min():.0f}, max {c.max():.0f}, median {c.median():.0f}")

**Result: partially reproduced.** The reaches tables hold 211 to 996 events per subject
(median 770.5), and 208 to 986 (median 748) remain after segment rejection. Under both readings
the largest count is within tolerance of 947, but the smallest and the median are above the
paper's 151 and 640. Twelve of 55 files hold 199 or 200 events, the Methods' cap per day before
visual inspection removed 17.8 +/- 9.9% of events as false positives; taking 17.8% off every
subject gives 173 to 819 (median 633). The dandiset marks no events as false positives.

## features-differ-subjects

> All ten behavioural and environmental features differ across subjects (Kruskal-Wallis,
> p < 0.001 for every feature).

In [ ]:
rows = []
for f in ajile.FEATURES:
    groups = [g[f].to_numpy() for _, g in ev.groupby("subject")]
    h, p = stats.kruskal(*groups)
    rows.append({"feature": f, "H": h, "p": p, "eta2_H": (h - len(groups) + 1) / (len(ev) - len(groups))})
kw = pd.DataFrame(rows)
print(kw.to_string(index=False, formatters={"p": "{:.2e}".format, "H": "{:.1f}".format, "eta2_H": "{:.3f}".format}))
print("Features with p < 0.001:", int((kw.p < 0.001).sum()), "of 10")

**Result: reproduced.** All ten features differ across subjects with p < 0.001 (the largest p
is 1.2e-34, for duration). The effect sizes range from 0.022 (duration) to 0.250 (onset speed).

## days-one-dominant

> The number of events was fairly consistent across days, except in subjects 04, 05 and 09,
> who each had one day representing most of their events.

In [ ]:
share = ev.groupby(["subject", "day"]).size().unstack()
top = (share.max(axis=1) / share.sum(axis=1)).rename("largest_day_share")
print(share.fillna(0).astype(int).assign(largest_day_share=top.round(3)).to_string())
over_half = sorted(top[top > 0.5].index)
top3 = sorted(top.sort_values(ascending=False).index[:3])
print("\n(a) subjects with one day above half their events:", over_half)
print("(b) the three subjects with the largest one-day share:", top3)

**Result: reproduced.** Under reading (b), subjects 04, 05 and 09 have the three largest shares
of their events on one day (0.381, 0.610 and 0.517; the next is 0.308). Under reading (a) only
subjects 05 and 09 have more than half on one day.

## duration-magnitude-small

> Most reaches last less than 2 s and cover less than 200 pixels.

In [ ]:
frac = ev.groupby("subject").agg(dur_below_2s=("duration", lambda d: (d < 2).mean()),
                                 mag_below_200px=("magnitude", lambda m: (m < 200).mean()),
                                 median_duration=("duration", "median"), median_magnitude=("magnitude", "median"))
print(frac.round(3).to_string())
print("\nAll events: below 2 s", round(float((ev.duration < 2).mean()), 3), "; below 200 px", round(float((ev.magnitude < 200).mean()), 3))
print("Subjects where both are above half:", int(((frac.dur_below_2s > 0.5) & (frac.mag_below_200px > 0.5)).sum()), "of 12")

In [ ]:
# Variant: only events lasting 0.5 to 4 s, the range the Methods keep.
in_range = ev.duration.between(0.5, 4)
print("Events lasting 0.5 to 4 s:", int(in_range.sum()), "of", len(ev))
print("Of these: below 2 s", round(float((ev.duration[in_range] < 2).mean()), 3),
      "; below 200 px", round(float((ev.magnitude[in_range] < 200).mean()), 3))

**Result: reproduced.** Over all events, 91.7% last less than 2 s and 91.5% cover less than
200 pixels, and both fractions are above half in all 12 subjects (the lowest is 0.590, for
magnitude in subject 08). Keeping only the 7760 events that last 0.5 to 4 s, the range the
Methods keep, gives 91.4% and 91.7%.

## angle-bimodal

> Reach angle distributions tend to be bimodal, with peaks at +/-90 degrees and few
> side-to-side reaches.

In [ ]:
bins = np.arange(-90, 91, 15)
hist = np.histogram(ev.angle, bins=bins)[0]
print("Folded angle, events per 15-degree bin from -90 (down) to 90 (up):", hist.tolist())
print("Two most populated bins:", sorted(np.argsort(hist)[-2:].tolist()), "(outer bins are 0 and 11)")
per = ev.groupby("subject").angle.agg(steep=lambda a: (np.abs(a) > 60).mean(), flat=lambda a: (np.abs(a) < 30).mean(),
                                       up=lambda a: (a > 60).mean(), down=lambda a: (a < -60).mean())
print(per.round(3).to_string())
print("Subjects with more events above |60| than below |30| degrees:", int((per.steep > per.flat).sum()), "of 12")
fig, ax = plt.subplots(figsize=(5, 2.5))
ax.hist(ev.angle, bins=bins, color="0.4")
ax.set_xlabel("folded reach angle (deg, up positive)")
ax.set_ylabel("events")
plt.tight_layout()
plt.show()
print("Plotted range:", round(float(ev.angle.min()), 1), "to", round(float(ev.angle.max()), 1), "deg; none outside the axes")

In [ ]:
# The stored angle before folding, per subject (image coordinates, y down: -90 is straight up,
# 90 straight down, 0 and +/-180 sideways), in 30-degree bins centred on multiples of 30.
edges = np.arange(-195, 196, 30)
wrapped = (ev.angle_stored + 195) % 360 - 195
centres = (edges[:-1] + 15).astype(int)
h = pd.DataFrame([np.histogram(g, bins=edges)[0] for _, g in wrapped.groupby(ev.subject)],
                 index=subjects, columns=centres)
h[180] = h[180] + h[-180]
h = h.drop(columns=-180)
print("Stored angle, events per 30-degree bin, by bin centre:")
print(h.to_string())
updown = pd.DataFrame({"up_-120_to_-60": [(g.between(-120, -60)).mean() for _, g in wrapped.groupby(ev.subject)],
                       "down_60_to_120": [(g.between(60, 120)).mean() for _, g in wrapped.groupby(ev.subject)],
                       "sideways_within_30_of_0_or_180": [((g.abs() < 30) | (g.abs() > 150)).mean() for _, g in wrapped.groupby(ev.subject)]},
                      index=subjects)
print(updown.round(3).to_string())
print("Subjects with more upward (-120 to -60) than downward (60 to 120) stored angles:",
      int((updown.iloc[:, 0] > updown.iloc[:, 1]).sum()), "of 12")

**Result: not reproduced.** Over all events the two most populated bins are 60 to 75 and 75 to
90 degrees (1304 and 1605 events): upward reaches dominate. The downward outer bin holds 514
events, more than its neighbour (487) but fewer than six other bins, so the pooled
part of the criterion fails. In 11 of 12 subjects steep reaches (|angle| above 60 degrees)
outnumber flat ones (below 30), and in every subject upward steep reaches outnumber downward
ones. The stored angles, before folding, show the same skew in every subject: 22.6% to 46.7% of
events lie within 30 degrees of straight up and 7.0% to 20.9% within 30 degrees of straight
down, so the skew is in the dandiset's values and not made by the folding.

## feature-correlations

> Reach magnitude correlates with duration (r = 0.26) and onset speed (r = 0.56), duration with
> bimanual overlap (r = 0.48), and the bimanual features with each other (0.51, 0.50, 0.61).

In [ ]:
pairs = [("magnitude", "duration", 0.26), ("magnitude", "onset_speed", 0.56), ("duration", "bimanual_overlap", 0.48),
         ("bimanual_overlap", "bimanual_ratio", 0.51), ("bimanual_class", "bimanual_ratio", 0.50),
         ("bimanual_overlap", "bimanual_class", 0.61)]
rows = []
for a, b, paper in pairs:
    r_all = ev[a].corr(ev[b])
    r_keep = ev[pw.keep.to_numpy()][a].corr(ev[pw.keep.to_numpy()][b])
    rows.append({"pair": f"{a} / {b}", "paper": paper, "r": r_all, "within_10pct": abs(r_all - paper) <= 0.1 * paper,
                 "r_kept_events": r_keep, "spearman": stats.spearmanr(ev[a], ev[b])[0]})
corr = pd.DataFrame(rows)
print(corr.round(3).to_string(index=False))
print("Within tolerance:", int(corr.within_10pct.sum()), "of 6; all positive:", bool((corr.r > 0).all()))

In [ ]:
# Variant: only events lasting 0.5 to 4 s, the range the Methods keep.
in_range = ev.duration.between(0.5, 4).to_numpy()
r_range = [ev[in_range][a].corr(ev[in_range][b]) for a, b, _ in pairs]
print(pd.DataFrame({"pair": corr.pair, "paper": corr.paper, "r_0.5_to_4_s": np.round(r_range, 3),
                    "within_10pct": [abs(r - p) <= 0.1 * p for r, (_, _, p) in zip(r_range, pairs)]}).to_string(index=False))

**Result: partially reproduced.** All six correlations are positive and five are within 10% of
the paper's values. Magnitude and duration correlate at 0.231, just outside 0.26 +/- 0.026. The
events left after segment rejection give the same values within 0.004. Keeping only the events
that last 0.5 to 4 s, the range the Methods keep, all six are within tolerance (magnitude and
duration 0.255); this variant does not change the outcome, which the criterion fixes on all
events.

## bimanual-skew

> Bimanual ratio and overlap are skewed toward unimanual movements, less so in subjects 02, 04
> and 05, while the bimanual class is mostly bimanual.

In [ ]:
print("All events: median bimanual ratio", round(float(ev.bimanual_ratio.median()), 3),
      "; median overlap", round(float(ev.bimanual_overlap.median()), 3), "s against median duration",
      round(float(ev.duration.median()), 3), "s; fraction bimanual class", round(float(ev.bimanual_class.mean()), 3))
bs = ev.groupby("subject").agg(median_ratio=("bimanual_ratio", "median"), median_overlap=("bimanual_overlap", "median"),
                               bimanual_class=("bimanual_class", "mean")).sort_values("median_ratio", ascending=False)
print(bs.round(3).to_string())
print("Four subjects with the highest median ratio:", sorted(bs.index[:4]))

**Result: reproduced.** Over all events the median bimanual ratio is 0.137, the median overlap
0.467 s against a median duration of 1.1 s, and 72.6% of events are of class bimanual.
Subjects 04, 05 and 02 are among the four with the highest median ratio (with 07).

## projection-link

> The paper's projection matrices can be matched to the dandiset's electrodes, and regions
> with an average electrode density above 3 are the paper's eight.

In [ ]:
rng_link = np.random.default_rng(55005)


def closeness_score(xyz, w, cen):
    rs = []
    for j, roi in enumerate(ajile.ROIS):
        d = np.linalg.norm(xyz - cen.loc[roi].to_numpy(float), axis=1)
        ok = np.isfinite(d) & (w[:, j] > 0)
        rs.append(stats.spearmanr(-d[ok], w[ok, j])[0])
    return float(np.median(rs))


density = sum(proj[s].iloc[0] for s in subjects) / len(subjects)
chosen = sorted(density[density > 3].index)
print("Regions with average density above 3:", chosen)
print("Same as the paper's eight:", chosen == sorted(ajile.ROIS))
rows = []
for sub in subjects:
    e = first[sub]["electrodes"]
    m = proj[sub]
    w = m.iloc[1:][ajile.ROIS].to_numpy(float)
    extra = w[len(e):]
    xyz = e[["x", "y", "z"]].to_numpy(float).copy()
    xyz[:, 0] = -np.abs(xyz[:, 0])                 # all electrodes mirrored to the left, as in the code
    w = w[:len(e)]
    score = closeness_score(xyz, w, centroids[sub])
    null = [closeness_score(xyz[rng_link.permutation(len(xyz))], w, centroids[sub]) for _ in range(20)]
    zero_rows = np.flatnonzero((m.iloc[1:len(e) + 1] == 0).all(axis=1).to_numpy())
    rows.append({"subject": sub, "matrix_rows": len(m) - 1, "electrodes": len(e), "extra_rows_all_zero": bool((extra == 0).all()),
                 "score": score, "shuffled_max": max(null), "zero_weight_rows": f"{zero_rows.min()}-{zero_rows.max()}" if len(zero_rows) else "",
                 "rows_without_xyz": ",".join(map(str, np.flatnonzero(e.x.isna().to_numpy())[[0, -1]])) if e.x.isna().any() else "",
                 "rows_without_R2": ",".join(map(str, np.flatnonzero(e.low_freq_R2.isna().to_numpy())[[0, -1]])) if e.low_freq_R2.isna().any() else ""})
link = pd.DataFrame(rows)
print(link.round(3).to_string(index=False))
link_ok = (link.extra_rows_all_zero & (link.score > 0.5) & (link.score > link.shuffled_max))
print("Subjects where the link holds:", int(link_ok.sum()), "of 12")

**Result: reproduced.** In all 12 subjects the matrix has a row for each electrode of the
dandiset's table (subject 01's four extra rows, past the table, are zeros), the eight regions
with density above 3 are the paper's, and an electrode's weight for a region follows its
closeness to the region's centroid (median rank correlation 0.917 to 0.967, against at most
0.143 with shuffled rows). Rows of zero weight are electrodes without coordinates in subject 03
(rows 80 to 95), but in subject 12 (rows 72 to 79) they have coordinates in the dandiset, and
that subject's electrodes without stored R2 are others (rows 116 to 123).

## group-pattern-rois

> At movement onset, low-frequency power decreases and high-frequency power increases, with the
> largest changes in the precentral, postcentral and inferior parietal regions and minimal
> changes in the three temporal regions.

Set A: five events per file (seeded), all electrodes carrying 99% of every region's weight.
Events dropped by the segment rejection of set P are left out.

In [ ]:
keep_key = set(map(tuple, pw[pw.keep][["subject", "day", "event"]].to_numpy()))
sub_tfr = {}
n_used = {}
for s, k, o in zip(sessions, sub_idx, out_a):
    n = len(k)
    good = np.array([(s["subject"], s["day"], int(e)) in keep_key for e in k]) & ~o["bad_window"][:n]
    sub_tfr.setdefault(s["subject"], []).append(o["roi_tfr"][:n][good][:, :8])
    n_used[s["subject"]] = n_used.get(s["subject"], 0) + int(good.sum())
per_subject_tfr = np.array([np.median(np.concatenate(sub_tfr[sub]), axis=0) for sub in subjects])   # subjects x regions x f x t
group_tfr = per_subject_tfr.mean(axis=0)
bands_sub = ajile.band_values(per_subject_tfr)          # subjects x regions x 2
bands_grp = ajile.band_values(group_tfr[None])[0]
print("Events used per subject:", n_used)
g = pd.DataFrame(bands_grp, index=[ajile.ROI_NAMES[r] for r in ajile.ROIS], columns=["LFB", "HFB"])
g["abs_sum"] = g.LFB.abs() + g.HFB.abs()
g["subjects_LFB_below_0"] = (bands_sub[:, :, 0] < 0).sum(axis=0)
g["subjects_HFB_above_0"] = (bands_sub[:, :, 1] > 0).sum(axis=0)
print(g.round(3).to_string())
sm = ["precentral", "postcentral", "inferior parietal"]
tm = ["superior temporal", "middle temporal", "inferior temporal"]
signs = bool((g.loc[sm, "LFB"] < 0).all() and (g.loc[sm, "HFB"] > 0).all())
ranking = bool(g.loc[sm, "abs_sum"].min() > g.loc[tm, "abs_sum"].max())
print("LFB < 0 and HFB > 0 in the three sensorimotor regions:", signs,
      "; each larger than every temporal region:", ranking)

fig, axes = plt.subplots(2, 4, figsize=(11, 4.5), sharex=True, sharey=True)
for j, ax in enumerate(axes.flat):
    im = ax.pcolormesh(ajile.TIMES, ajile.FREQS, group_tfr[j], cmap="RdBu_r", vmin=-1, vmax=1, shading="nearest")
    ax.set_title(ajile.ROI_NAMES[ajile.ROIS[j]], fontsize=9)
    ax.axvline(0, color="k", lw=0.8, ls="--")
for ax in axes[:, 0]:
    ax.set_ylabel("Hz")
for ax in axes[1]:
    ax.set_xlabel("s from movement onset")
fig.colorbar(im, ax=axes, label="dB from baseline", shrink=0.8)
plt.show()
print("Plotted values range from", round(float(group_tfr.min()), 2), "to", round(float(group_tfr.max()), 2),
      "dB; the colour scale saturates at +/-1 dB for",
      int((np.abs(group_tfr) > 1).sum()), "of", group_tfr.size, "bins")

In [ ]:
# Variant: precentral value with the full projection (set A) against the 95% electrode set of
# set P, on the same events; and the set P precentral result for those events.
rows = []
for s, k, o in zip(sessions, sub_idx, out_a):
    n = len(k)
    full = ajile.band_values(o["roi_tfr"][:n, 0])
    trunc = ajile.band_values(o["roi_tfr"][:n, 8])
    rows.append(pd.DataFrame({"lfb_full": full[:, 0], "lfb_95": trunc[:, 0], "hfb_full": full[:, 1], "hfb_95": trunc[:, 1]}))
tv = pd.concat(rows)
print(f"Precentral, full weights against the 95% electrode set, same events (n = {len(tv)}):")
print(f"  LFB r = {tv.lfb_full.corr(tv.lfb_95):.3f}, median difference {(tv.lfb_full - tv.lfb_95).median():.3f} dB; "
      f"HFB r = {tv.hfb_full.corr(tv.hfb_95):.3f}, median difference {(tv.hfb_full - tv.hfb_95).median():.3f} dB")

In [ ]:
# Uncertainty: events resampled with replacement within each subject (1000 resamples).
rng_grp = np.random.default_rng(55007)
ev_tfr = [np.concatenate(sub_tfr[sub]) for sub in subjects]
boot_bands = []
for _ in range(1000):
    pst = np.array([np.median(x[rng_grp.integers(0, len(x), len(x))], axis=0) for x in ev_tfr])
    boot_bands.append(ajile.band_values(pst.mean(axis=0)[None])[0])
boot_bands = np.array(boot_bands)                     # resamples x regions x 2
names = [ajile.ROI_NAMES[r] for r in ajile.ROIS]
abs_b = np.abs(boot_bands).sum(axis=2)
ci_tab = pd.DataFrame({"LFB_2.5": np.percentile(boot_bands[:, :, 0], 2.5, axis=0),
                       "LFB_97.5": np.percentile(boot_bands[:, :, 0], 97.5, axis=0),
                       "HFB_2.5": np.percentile(boot_bands[:, :, 1], 2.5, axis=0),
                       "HFB_97.5": np.percentile(boot_bands[:, :, 1], 97.5, axis=0),
                       "abs_sum_2.5": np.percentile(abs_b, 2.5, axis=0),
                       "abs_sum_97.5": np.percentile(abs_b, 97.5, axis=0)}, index=names)
print(ci_tab.round(3).to_string())
smi = [names.index(r) for r in sm]
tmi = [names.index(r) for r in tm]
holds = ((boot_bands[:, smi, 0] < 0).all(axis=1) & (boot_bands[:, smi, 1] > 0).all(axis=1)
         & (abs_b[:, smi].min(axis=1) > abs_b[:, tmi].max(axis=1)))
print("Fraction of resamples in which both parts of the criterion hold:", round(float(holds.mean()), 3))

**Result: reproduced.** Averaged over subjects, LFB power falls and HFB power rises from 0 to
0.5 s in the precentral (-0.549 and 0.487 dB), postcentral (-0.793 and 0.596 dB) and inferior
parietal (-0.879 and 0.280 dB) regions, and the summed change in each (1.036 to 1.389 dB) is
larger than in any temporal region (at most 0.337 dB). This rests on 15 to 25 events per
subject. Resampling the events within each subject, both parts of the criterion hold in 99.5%
of 1000 resamples; the 95% intervals of the summed change are 0.605 to 1.722 dB in the three
sensorimotor regions and reach at most 0.573 dB in the temporal ones. Restricting the precentral projection to the electrodes of set P changes it little
(r = 0.999 in both bands).

## precentral-subject-ci

> Precentral LFB and HFB power from 0 to 0.5 s differ between subjects as given by their 95%
> confidence intervals in Table 1.

In [ ]:
TABLE1 = {  # subject: (LFB low, LFB high, HFB low, HFB high), Table 1
    "01": (-1.87, -1.29, 0.03, 0.23), "02": (-1.12, -0.31, 0.07, 0.64), "03": (-2.11, -1.73, 1.39, 1.67),
    "04": (-0.17, 0.01, 0.32, 0.52), "05": (-0.7, -0.38, 0.2, 0.48), "06": (-1.63, -1.11, 2.02, 2.37),
    "07": (-0.84, -0.51, 1.14, 1.34), "08": (-0.3, 0.04, 0.36, 0.63), "09": (-0.57, -0.08, -0.11, 0.23),
    "10": (-0.87, -0.35, 0.01, 0.3), "11": (-2.3, -1.86, 0.09, 0.34), "12": (-0.66, -0.44, 0.06, 0.2)}
rng_boot = np.random.default_rng(55004)
kept = pw[pw.keep]
rows = []
for sub in subjects:
    g = kept[kept.subject == sub]
    for band, col, lo, hi in (("LFB", "lfb", TABLE1[sub][0], TABLE1[sub][1]), ("HFB", "hfb", TABLE1[sub][2], TABLE1[sub][3])):
        x = g[col].to_numpy()
        boot = rng_boot.choice(x, (2000, len(x)))
        rows.append({"subject": sub, "band": band, "n": len(x), "paper_lo": lo, "paper_hi": hi,
                     "mean": x.mean(), "median": np.median(x),
                     "mean_ci": f"{np.percentile(boot.mean(1), 2.5):.2f} to {np.percentile(boot.mean(1), 97.5):.2f}",
                     "median_ci": f"{np.percentile(np.median(boot, 1), 2.5):.2f} to {np.percentile(np.median(boot, 1), 97.5):.2f}",
                     "mean_in": lo <= x.mean() <= hi, "median_in": lo <= np.median(x) <= hi})
ci = pd.DataFrame(rows)
print(ci.round(2).to_string(index=False))
print("Inside the paper's interval: mean", int(ci.mean_in.sum()), "of 24; median", int(ci.median_in.sum()), "of 24")
for band in ("LFB", "HFB"):
    c = ci[ci.band == band]
    centre = (c.paper_lo + c.paper_hi) / 2
    print(f"{band}: Spearman r across subjects, median against interval centre {stats.spearmanr(c['median'], centre)[0]:.3f}, "
          f"mean against centre {stats.spearmanr(c['mean'], centre)[0]:.3f}")

In [ ]:
# Variants: without the segment rejection; without the three late-starting files.
for label, mask in (("no segment rejection", ~pw.bad_window),
                    ("without the three late-starting files", pw.keep & ~pw.set_index(["subject", "day"]).index.isin(
                        [(s["subject"], s["day"]) for s in sessions if s["ecog_offset"] > 0]))):
    for stat in ("mean", "median"):
        g = pw[mask].groupby("subject")[["lfb", "hfb"]].agg(stat)
        inside = sum(int(TABLE1[s][0] <= g.loc[s, "lfb"] <= TABLE1[s][1]) + int(TABLE1[s][2] <= g.loc[s, "hfb"] <= TABLE1[s][3])
                     for s in subjects)
        print(f"{label}: {stat}s inside the paper's interval for {inside} of 24")


**Result: reproduced.** The subject means fall inside the paper's 95% intervals for 22 of the
24 subject and band values (medians for 20). The two means outside are subject 06 HFB (1.99
against 2.02 to 2.37 dB) and subject 10 LFB (-0.30 against -0.87 to -0.35 dB). Across subjects
the values follow the interval centres (Spearman r 0.846 to 0.974). Without segment rejection
22 means and 21 medians fall inside, and without the three late-starting files 20 means and
19 medians.

## precentral-subjects-differ

> Precentral LFB and HFB power from 0 to 0.5 s differ across subjects (Kruskal-Wallis,
> p < 0.001 for both bands).

In [ ]:
for band in ("lfb", "hfb"):
    groups = [g[band].to_numpy() for _, g in kept.groupby("subject")]
    h, p = stats.kruskal(*groups)
    print(f"{band.upper()}: H = {h:.1f}, p = {p:.2e}, eta2_H = {(h - 11) / (len(kept) - 12):.3f}, n = {len(kept)}")
print("Paper's code printed p = 2.1e-159 (LFB) and 2.4e-278 (HFB).")

**Result: reproduced.** Precentral power differs across subjects in both bands (H = 802.1 for
LFB and 1243.5 for HFB over 7964 events; p is below the smallest number printed, so below
0.001).

## precentral-day-effects

> Precentral power depends on the recording day in several subjects; significant in LFB for
> 03, 05, 06, 07, 08 and 11, and in HFB for 03, 04, 07, 08 and 10.

In [ ]:
PAPER_DAY_SIG = {"lfb": {"03", "05", "06", "07", "08", "11"}, "hfb": {"03", "04", "07", "08", "10"}}


def day_tests(frame):
    rows = []
    for sub in subjects:
        g = frame[frame.subject == sub]
        for band in ("lfb", "hfb"):
            p = stats.kruskal(*[d[band].to_numpy() for _, d in g.groupby("day")])[1]
            rows.append({"subject": sub, "band": band, "p": p, "significant": p < 0.05,
                         "paper": sub in PAPER_DAY_SIG[band]})
    t = pd.DataFrame(rows)
    t["agree"] = t.significant == t.paper
    return t


dt = day_tests(kept)
print(dt.pivot(index="subject", columns="band", values=["p", "significant", "paper"]).to_string(
    formatters={("p", "lfb"): "{:.2e}".format, ("p", "hfb"): "{:.2e}".format}))
print("Calls that agree with the paper:", int(dt.agree.sum()), "of 24; significant subjects, LFB",
      int(dt[dt.band == "lfb"].significant.sum()), ", HFB", int(dt[dt.band == "hfb"].significant.sum()))
for label, mask in (("no segment rejection", ~pw.bad_window), ):
    v = day_tests(pw[mask])
    print(f"Variant, {label}: {int(v.agree.sum())} of 24 agree")
day_means = kept.groupby(["subject", "day"])[["lfb", "hfb"]].median().round(2)
print("\nMedian per subject and day (dB):")
print(day_means.unstack("day").to_string())

**Result: reproduced.** 23 of the 24 subject and band calls agree with the paper's: 5 subjects
have a significant day effect in each band. The one difference is subject 05 in LFB (p =
0.054, where the paper reports 0.013). The same 23 agree without segment rejection.

## baseline-reduces-day-variability

> Baseline subtraction reduces the variability of LFB and HFB power across recording days, but
> clear variability remains.

In [ ]:
rows = []
for sub in subjects:
    g = kept[kept.subject == sub].groupby("day")
    rows.append({"subject": sub,
                 "lfb_sd_subtracted": g.lfb.mean().std(), "lfb_sd_raw": g.lfb_raw.mean().std(),
                 "hfb_sd_subtracted": g.hfb.mean().std(), "hfb_sd_raw": g.hfb_raw.mean().std()})
bv = pd.DataFrame(rows).set_index("subject")
print(bv.round(3).to_string())
for band in ("lfb", "hfb"):
    lower = (bv[f"{band}_sd_subtracted"] < bv[f"{band}_sd_raw"]).sum()
    print(f"{band.upper()}: lower with subtraction in {lower} of 12 subjects; median ratio "
          f"{(bv[f'{band}_sd_subtracted'] / bv[f'{band}_sd_raw']).median():.3f}; all above 0: {bool((bv[f'{band}_sd_subtracted'] > 0).all())}")

In [ ]:
# Uncertainty: events resampled with replacement within each subject and day (1000 resamples).
rng_bv = np.random.default_rng(55008)
cols = ["lfb", "lfb_raw", "hfb", "hfb_raw"]
per_day = {sub: [g[cols].to_numpy() for _, g in kept[kept.subject == sub].groupby("day")] for sub in subjects}
diff = {band: np.empty((1000, len(subjects))) for band in ("lfb", "hfb")}
for b in range(1000):
    for j, sub in enumerate(subjects):
        m = np.array([d[rng_bv.integers(0, len(d), len(d))].mean(axis=0) for d in per_day[sub]])
        sdv = m.std(axis=0, ddof=1)
        diff["lfb"][b, j] = sdv[0] - sdv[1]
        diff["hfb"][b, j] = sdv[2] - sdv[3]
for band in ("lfb", "hfb"):
    d = diff[band]
    t = pd.DataFrame({"sd_subtracted_minus_raw_2.5": np.percentile(d, 2.5, axis=0),
                      "sd_subtracted_minus_raw_97.5": np.percentile(d, 97.5, axis=0),
                      "fraction_lower": (d < 0).mean(axis=0)}, index=subjects)
    print(band.upper())
    print(t.round(3).to_string())
    n_lower = (d < 0).sum(axis=1)
    print(f"{band.upper()}: subjects whose interval lies below 0: {int((t.iloc[:, 1] < 0).sum())} of 12; "
          f"subjects lower per resample, 2.5th to 97.5th percentile: {np.percentile(n_lower, 2.5):.0f} to "
          f"{np.percentile(n_lower, 97.5):.0f}; fraction of resamples with at least 9: {(n_lower >= 9).mean():.3f}")

**Result: reproduced.** The SD across days is lower with baseline subtraction in 9 of 12
subjects for LFB and in 12 of 12 for HFB (median ratios 0.216 and 0.291), and it stays above 0
in every subject (smallest 0.066 dB for LFB and 0.032 dB for HFB). The LFB count is at the
criterion's threshold: resampling the events within each subject and day, it ranges from 8 to
11 (95% of 1000 resamples) and is at least 9 in 90.0% of them, and the interval of the
difference lies below 0 in 8 subjects. For HFB the count ranges from 10 to 12 and is at least 9
in every resample.

## Regression (Figure 6)

For each subject, the up to eight good electrodes of set P with the largest precentral weight;
LFB and HFB power of each electrode (baseline-subtracted, median over 0 to 0.5 s and the band)
for the kept events; the code's design matrix and forward selection; Huber regression over 40
random day-balanced splits with 90% of events for training (the Methods; the code uses 80%, shown
as a variant with 10 splits). The paper uses 200 splits. Set P holds only 5 good electrodes in
subject 02 and 7 in subject 06, so 92 electrodes are tested, not 96. As in the code, forward
selection runs once on all events; a variant runs it on the training events of each split (10
splits, 90/10), as the Methods' order of steps implies.

In [ ]:
reg_tasks, reg_keys, sit_tasks, sit_keys = [], [], [], []
for sub in subjects:
    ss = [(s, t, o) for s, t, o in zip(sessions, tasks_p, out_p) if s["subject"] == sub]
    ch = ss[0][1]["channels"]
    good0 = first[sub]["electrodes"].good.to_numpy()
    order = [c for c in ch[np.argsort(-w_all[sub][ch, 0])] if good0[c]][:8]
    bands = np.concatenate([o["bands"] for _, _, o in ss])           # events x channels x 2
    mask = pw[pw.subject == sub].keep.to_numpy()
    evs = ev[ev.subject == sub][mask].reset_index(drop=True)
    x, names, groups = ajile.design(evs)
    for c in order:
        j = int(np.flatnonzero(ch == c)[0])
        for b, band in enumerate(("LFB", "HFB")):
            for per_train, n_splits, label in ((0.9, 40, "methods"), (0.8, 10, "code")):
                reg_tasks.append({"x": x, "y": bands[mask, j, b], "groups": groups, "days": evs.day.to_numpy(),
                                  "per_train": per_train, "n_splits": n_splits,
                                  "seed": 55003 + len(reg_tasks)})
                reg_keys.append((sub, int(c), band, label))
            # Variant: forward selection inside each training split, with seeds of its own.
            sit_tasks.append({"x": x, "y": bands[mask, j, b], "groups": groups, "days": evs.day.to_numpy(),
                              "per_train": 0.9, "n_splits": 10, "select_in_train": True,
                              "seed": 56003 + len(sit_tasks)})
            sit_keys.append((sub, int(c), band, "select_in_train"))
started = time.time()
reg_out = ajile.run_regressions(reg_tasks + sit_tasks)
reg_tasks, reg_keys = reg_tasks + sit_tasks, reg_keys + sit_keys
print(f"{len(reg_tasks)} models, {sum(r['n_fits'] for r in reg_out)} Huber fits in {time.time() - started:.0f} s; "
      f"fits that did not converge: {sum(r['unconverged'] for r in reg_out)}")
reg = pd.DataFrame([{"subject": k[0], "electrode": k[1], "band": k[2], "split": k[3], "r2_test": r["r2_test"],
                     "r2_train": r["r2_train"], "coef_angle": r["coef_angle"],
                     **{f"dm_{f}": v for f, v in r["delta_methods"].items()},
                     **{f"dc_{f}": v for f, v in r["delta_code"].items()},
                     **{f"ret_{f}": v for f, v in r["retained"].items()}} for k, r in zip(reg_keys, reg_out)])
main = reg[reg.split == "methods"]
sit = reg[reg.split == "select_in_train"]
print("Events with two time-of-day columns set (at exactly 8 or 16 h):", int(ev.time_of_day.isin([8.0, 16.0]).sum()), "of", len(ev))
print("Electrodes tested per subject:", main[main.band == "LFB"].groupby("subject").size().to_dict())

## regression-r2-max

> Regression models explain at most 25% of the variance of withheld events.

In [ ]:
print("Mean test R2 over splits, tested electrodes:")
print(main.groupby("band").r2_test.describe().round(3).to_string())
print("Largest:", round(float(main.r2_test.max()), 3), "at", main.loc[main.r2_test.idxmax(), ["subject", "electrode", "band"]].to_dict())
print("Code's split (80/20, 10 splits): largest", round(float(reg[reg.split == "code"].r2_test.max()), 3))
stored = []
for _, r in main.iterrows():
    e = first[r.subject]["electrodes"]
    stored.append(e.low_freq_R2.iloc[r.electrode] if r.band == "LFB" else e.high_freq_R2.iloc[r.electrode])
main = main.assign(stored_r2=stored)
for band in ("LFB", "HFB"):
    m = main[main.band == band]
    print(f"{band}: recomputed against stored R2 on the same electrodes, Pearson r = {m.r2_test.corr(m.stored_r2):.3f}, "
          f"median recomputed {m.r2_test.median():.3f}, median stored {m.stored_r2.median():.3f}")
all_stored = pd.concat([first[s]["electrodes"][["low_freq_R2", "high_freq_R2"]] for s in subjects])
print("Stored R2 over all electrodes: largest LFB", round(float(all_stored.low_freq_R2.max()), 3),
      ", HFB", round(float(all_stored.high_freq_R2.max()), 3))

In [ ]:
# Variant: forward selection on the training events of each split (10 splits, 90/10).
print("Selection inside each training split: largest mean test R2", round(float(sit.r2_test.max()), 3),
      "; median by band", sit.groupby("band").r2_test.median().round(3).to_dict(),
      "; against the main analysis", main.groupby("band").r2_test.median().round(3).to_dict())
print("Mean change in test R2 per model (in-train minus main), by band:",
      (sit.set_index(["subject", "electrode", "band"]).r2_test - main.set_index(["subject", "electrode", "band"]).r2_test)
      .groupby("band").mean().round(4).to_dict())

**Result: reproduced.** The largest recomputed mean test R2 is 0.141 (subject 03, LFB), within
the bound of 0.25. The R2 values stored in the dandiset, which cover every electrode, reach
0.255 (HFB); on the tested electrodes they correlate with the recomputed ones (r = 0.865 for
LFB and 0.776 for HFB). With forward selection inside each training split, the largest mean
test R2 is 0.140, and the test R2 of a model is lower by 0.033 (LFB) and 0.035 (HFB) on average
(10 splits against 40).

## regression-r2-sensorimotor

> The models with the largest R2 on withheld data are in frontoparietal sensorimotor areas.

On the R2 values stored in the dandiset for every electrode, projected onto the regions.

In [ ]:
rows = []
for sub in subjects:
    e = first[sub]["electrodes"]
    for band, col in (("LFB", "low_freq_R2"), ("HFB", "high_freq_R2")):
        r2 = e[col].to_numpy(float)
        good = e.good.to_numpy() & np.isfinite(r2)
        w = ajile.roi_weights(proj[sub], len(e), good)
        rows.append({"subject": sub, "band": band, **dict(zip([ajile.ROI_NAMES[r] for r in ajile.ROIS], np.nan_to_num(r2) @ w))})
pr = pd.DataFrame(rows)
reg_roi = pr.groupby("band")[[ajile.ROI_NAMES[r] for r in ajile.ROIS]].mean().T
print(reg_roi.round(4).to_string())
fp = ["precentral", "postcentral", "middle frontal", "inferior parietal", "supramarginal"]
for band in ("LFB", "HFB"):
    top = reg_roi[band].idxmax()
    print(f"{band}: highest region {top}; frontoparietal mean {reg_roi.loc[fp, band].mean():.4f}, "
          f"temporal mean {reg_roi.loc[tm, band].mean():.4f}")
print("Electrodes with stored R2 > 0.05: LFB", int((all_stored.low_freq_R2 > 0.05).sum()), ", HFB", int((all_stored.high_freq_R2 > 0.05).sum()))

In [ ]:
# The 20 good electrodes with the largest stored R2 in each band, with the region of their
# largest projection weight among the eight ("none" where all eight weights are 0).
rows = []
for sub in subjects:
    e = first[sub]["electrodes"]
    w = proj[sub].iloc[1:len(e) + 1][ajile.ROIS].to_numpy(float)
    for j in range(len(e)):
        if e.good.iloc[j]:
            rows.append({"subject": sub, "electrode": j, "LFB": e.low_freq_R2.iloc[j], "HFB": e.high_freq_R2.iloc[j],
                         "region": ajile.ROI_NAMES[ajile.ROIS[int(np.argmax(w[j]))]] if w[j].max() > 0 else "none",
                         "weight": w[j].max()})
el_r2 = pd.DataFrame(rows)
for band in ("LFB", "HFB"):
    top20 = el_r2.dropna(subset=[band]).nlargest(20, band)
    print(f"{band}, top 20 stored R2 ({top20[band].min():.3f} to {top20[band].max():.3f}); by region:", top20.region.value_counts().to_dict(),
          "; by subject:", top20.subject.value_counts().sort_index().to_dict())
    print(f"   frontoparietal {int(top20.region.isin(fp).sum())}, temporal {int(top20.region.isin(tm).sum())}, "
          f"none {int((top20.region == 'none').sum())} of 20")

**Result: reproduced.** On the stored R2 values, the region with the highest projected R2 is
the precentral for LFB and the postcentral for HFB, and the frontoparietal mean is higher than
the temporal mean in both bands. The projected values are all near or below 0 (-0.0192 to
0.0008), as only 24 (LFB) and 25 (HFB) electrodes have a stored R2 above 0.05. Of the 20 good electrodes
with the largest stored R2, 19 (LFB) and 16 (HFB) have their largest projection weight in a
frontoparietal region; for LFB they come from five subjects, 9 of them from subject 01.

## regression-angle-day-top

> Reach angle and recording day are the most explanatory features, especially for
> low-frequency power.

In [ ]:
well = main[main.band == "LFB"].set_index(["subject", "electrode"]).r2_test > 0.05
well_keys = set(well[well].index)
sel = main[[(s, e) in well_keys for s, e in zip(main.subject, main.electrode)]]
print("Electrodes with mean LFB test R2 > 0.05:", len(well_keys), "of", len(well), "; by subject:",
      pd.Series([k[0] for k in well_keys]).value_counts().sort_index().to_dict())
feats = ajile.REG_FEATURES
for prefix, label in (("dm_", "Methods (unshuffled test events)"), ("dc_", "code (shuffled test events)")):
    t = sel.groupby("band")[[prefix + f for f in feats]].mean().T
    t.index = feats
    print(f"\nMean Delta R2, {label}:")
    print(t.round(4).to_string())
    print("Two largest in LFB:", list(t["LFB"].sort_values(ascending=False).index[:2]),
          "; in HFB:", list(t["HFB"].sort_values(ascending=False).index[:2]))
t_all = main.groupby("band")[["dm_" + f for f in feats]].mean().T
t_all.index = feats
print("\nVariant, all tested electrodes (Methods):", "LFB top two", list(t_all["LFB"].sort_values(ascending=False).index[:2]))
t_code = reg[(reg.split == "code")].groupby("band")[["dc_" + f for f in feats]].mean().T
t_code.index = feats
print("Variant, code's split and scoring, all tested electrodes: LFB top two", list(t_code["LFB"].sort_values(ascending=False).index[:2]))

In [ ]:
# Variant: forward selection on the training events of each split, electrodes chosen by this
# variant's own mean LFB test R2 above 0.05.
well_v = sit[sit.band == "LFB"].set_index(["subject", "electrode"]).r2_test > 0.05
keys_v = set(well_v[well_v].index)
sel_v = sit[[(s, e) in keys_v for s, e in zip(sit.subject, sit.electrode)]]
t = sel_v.groupby("band")[["dm_" + f for f in feats]].mean().T
t.index = feats
print("Selection inside each training split: electrodes with LFB R2 > 0.05:", len(keys_v),
      "(", len(keys_v & well_keys), "shared with the main selection )")
print(t.round(4).to_string())
print("Two largest in LFB:", list(t["LFB"].sort_values(ascending=False).index[:2]))

**Result: reproduced.** On the 12 tested electrodes with mean LFB test R2 above 0.05 (6 in
subject 01, 4 in 03, 2 in 11), reach angle (0.0645) and day (0.0280) have the largest mean
Delta R2 for LFB, and also for HFB. The same holds with the code's split and scoring, and over
all tested electrodes. The well-fit electrodes come from three subjects only. With forward selection inside each
training split, 8 electrodes pass (6 of them in the main selection), and angle (0.0648) and day
(0.0376) again have the largest mean Delta R2 in LFB.

## regression-angle-coefficient

> Upward reaches are associated with decreased low-frequency and increased high-frequency
> power.

In [ ]:
c = sel.groupby("band").coef_angle.agg(["mean", "median", "count", lambda x: (x < 0).mean()])
c.columns = ["mean", "median", "n", "fraction_negative"]
print(c.round(4).to_string())
for band in ("LFB", "HFB"):
    x = sel[sel.band == band].coef_angle
    print(f"{band}: Wilcoxon against 0, p = {stats.wilcoxon(x)[1]:.3g}")
print("All tested electrodes:", main.groupby("band").coef_angle.mean().round(4).to_dict())

In [ ]:
# Variant: forward selection on the training events of each split, on that variant's electrodes.
print("Selection inside each training split: mean angle coefficient", sel_v.groupby("band").coef_angle.mean().round(4).to_dict(),
      "; fraction negative", sel_v.groupby("band").coef_angle.apply(lambda x: (x < 0).mean()).round(3).to_dict(),
      "; n", sel_v.groupby("band").size().to_dict())

**Result: reproduced.** On the same 12 electrodes the mean reach angle coefficient is -1.065
for LFB (negative on all 12, Wilcoxon p = 0.00049) and 0.2145 for HFB (positive on 11 of 12,
p = 0.012): upward reaches go with a deeper LFB decrease and a larger HFB increase. With forward
selection inside each training split, the means are -1.091 (LFB, negative on 8 of 8) and 0.277
(HFB, positive on 7 of 8).

## regression-angle-retained

> Reach angle is the feature most often retained by forward selection in sensorimotor areas.

In [ ]:
ret = main[["ret_" + f for f in feats]].mean()
ret.index = feats
print("Fraction of models (tested electrodes, both bands) retaining each feature:")
print(ret.sort_values(ascending=False).round(3).to_string())
print("\nBy band:")
rb = main.groupby("band")[["ret_" + f for f in feats]].mean().T
rb.index = feats
print(rb.round(3).to_string())

In [ ]:
# Variant: forward selection on the training events of each split (fraction of splits, then of models).
ret_v = sit[["ret_" + f for f in feats]].mean()
ret_v.index = feats
print("Selection inside each training split:")
print(ret_v.sort_values(ascending=False).round(3).to_string())

**Result: reproduced.** Forward selection retains reach angle in 64.1% of the models on the
tested electrodes, more than any other feature (next, onset speed at 38.6%), and in both
bands. With forward selection inside each training split, angle is retained in 61.8% of
splits and models, ahead of bimanual ratio (34.6%).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| subjects-days-hemispheres | dandiset | reproduced |
| demographics | dandiset | blocked: data missing |
| electrode-counts | dandiset | partially reproduced |
| bad-electrodes | dandiset | partially reproduced |
| event-counts | dandiset | partially reproduced |
| features-differ-subjects | dandiset | reproduced |
| days-one-dominant | dandiset | reproduced |
| duration-magnitude-small | dandiset | reproduced |
| angle-bimodal | dandiset | not reproduced |
| feature-correlations | dandiset | partially reproduced |
| bimanual-skew | dandiset | reproduced |
| projection-link | with inputs | reproduced |
| group-pattern-rois | with inputs | reproduced |
| precentral-subject-ci | with inputs | reproduced |
| precentral-subjects-differ | with inputs | reproduced |
| precentral-day-effects | with inputs | reproduced |
| baseline-reduces-day-variability | with inputs | reproduced |
| regression-r2-max | with inputs | reproduced |
| regression-r2-sensorimotor | with inputs | reproduced |
| regression-angle-day-top | with inputs | reproduced |
| regression-angle-coefficient | with inputs | reproduced |
| regression-angle-retained | with inputs | reproduced |

**How the analysis differs from the paper's.**

- Events are those of the dandiset's reaches tables. The paper's visual removal of false
  positives cannot be repeated, and the dandiset does not mark them; the segment rejection
  follows the Methods as far as they can be read (missing or flat data, mean power below 0 dB in
  microvolts squared, power at 116 and 121 Hz more than 3 SD above the subject's mean).
- In three files the event times are moved onto the ECoG clock (see the data check).
- Spectral power is computed here with the code's Morlet parameters, not with MNE, on 4 s
  windows read from the continuous ECoG, not on the paper's 10 s segments.
- Precentral power uses the electrodes that carry 95% of the precentral weight; the
  eight-region pattern uses five events per file. The regression uses up to eight electrodes per
  subject, 40 splits in place of 200, and the Methods' 90/10 split and Delta R2 scoring, with
  the code's 80/20 split and scoring as a variant; forward selection runs on all events, as in
  the code, with selection inside each training split shown as a variant. Subjects 02 and 06
  have only 5 and 7 tested electrodes.
- The sensorimotor location of the best regression models is judged on the R2 stored in the
  dandiset, not on a recomputation.

**What was found about the dandiset.**

- In three files (subject 01 day 7, 04 day 3, 06 day 3) the ECoG starts at
  `session_start_time`, 8.1 to 8.5 h after the start of the day, but its `starting_time` is 0,
  while the reaches table, the epochs and the pose count from the start of the day. The
  `ReachEvents` timestamps of these files are on the ECoG clock and disagree with the reaches
  table.
- `session_start_time` gives 19:00 at UTC-05:00 for the full-day files, while the sleep labels
  place the start of the file at local midnight.
- The events are those before the paper's visual removal of false positives, with no mark of
  which were removed; per subject counts are 211 to 996 against the paper's 151 to 947.
- Subject sex and age are not in the dandiset.
- `Bimanual_overlap` is in seconds, though described as a fraction; the reach angle's convention
  (image coordinates, y down) is not stated; pose is stated in meters with a NaN conversion but
  is in pixels; the ECG series carry the EOG description; electrode location is "unknown" and
  impedance missing for every electrode.
- Electrode groups described as depth occur in seven subjects, where the paper reports five;
  subject 03's 16 depth electrodes have no coordinates; in subject 12 the electrodes without
  stored R2 are not those without projection weight.
- One event (subject 08 day 4) has a `start_time` past the end of its 24 h file.
- The ECoG chunk length differs between files (84375 samples, but 54412 to 55820 in the three
  files that start later in the day), and over the network h5py takes about 0.4 s per
  chunk row to look chunks up; the notebook reads the chunk index directly.

In [ ]:
print(f"Notebook ran in {(time.time() - nb_started) / 60:.1f} min.")